In [1]:
import sys
import yaml
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root / "src"))

from bonn_thesis.openai_processing.soep_batch_manager import (
    check_status,
    download_batch_results,
    submit_batch,
)
from bonn_thesis.config import LINKEDIN_MATCHED_TO_SOEP_BLD, SRC

## Submit LinkedIn Batch to OpenAI

This cell submits the prepared LinkedIn data to OpenAI's batch API.

In [2]:
# Load experiment configuration
experiment_config_path = SRC / "openai_processing" / "configs" / "experiments" / "wage_linkedin_exp_14.yaml"
with open(experiment_config_path) as f:
    experiment_config = yaml.safe_load(f)

# Paths
jsonl_path = LINKEDIN_MATCHED_TO_SOEP_BLD / "openai_inputs" / "wage_linkedin_exp_14a.jsonl"
metadata_csv_path = LINKEDIN_MATCHED_TO_SOEP_BLD / "linkedin_metadata.csv"

# Submit batch
result = submit_batch(
    jsonl_path=str(jsonl_path),
    experiment_config=experiment_config,
    metadata_csv_path=str(metadata_csv_path),
)

# Save the batch ID for later use
batch_id = result["batch_id"]
print("\n✓ Batch submitted!")
print(f"Batch ID: {batch_id}")
print(f"Batch Name: {result['batch_name']}")

SUBMITTING BATCH: wage_linkedin_exp_14
Uploading file: /Users/willbackes/Documents/code/bonn_thesis/bld/data/linkedin_matched_to_soep/openai_inputs/wage_linkedin_exp_14a.jsonl
File uploaded successfully. File ID: file-BYxGLLvPngkYqS5E4SKRWb
Creating batch: wage_linkedin_exp_14
Batch created successfully. Batch ID: batch_69ac3b7c1d2481909a79bd49efedd1cf
Status: validating
Metadata updated in /Users/willbackes/Documents/code/bonn_thesis/bld/data/linkedin_matched_to_soep/linkedin_metadata.csv
SUBMISSION COMPLETE
Batch Name: wage_linkedin_exp_14
Batch ID: batch_69ac3b7c1d2481909a79bd49efedd1cf
File ID: file-BYxGLLvPngkYqS5E4SKRWb

✓ Batch submitted!
Batch ID: batch_69ac3b7c1d2481909a79bd49efedd1cf
Batch Name: wage_linkedin_exp_14


/Users/willbackes/Documents/code/bonn_thesis/src/bonn_thesis/openai_processing/soep_batch_manager.py:226: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'batch_69ac3b7c1d2481909a79bd49efedd1cf' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, key] = value
/Users/willbackes/Documents/code/bonn_thesis/src/bonn_thesis/openai_processing/soep_batch_manager.py:226: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2026-03-07T15:51:40.575854' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, key] = value


## Check Batch Status

Use this cell to check the status of your batch job. Replace `batch_id` with your actual batch ID from the submission step above.

In [3]:
# Replace with your batch ID from the submission step
# batch_id = "batch_xxxxxxxxxxxxx"

status_info = check_status(batch_id)

# Print status
current_status = status_info["status"]
print(f"\nCurrent status: {current_status}")

if current_status == "completed":
    print("\n✓ Batch is complete! Ready to download results.")
elif current_status in ["validating", "in_progress", "finalizing"]:
    print("\n⏳ Batch is still processing. Check back later.")
elif current_status == "failed":
    print("\n❌ Batch failed. Check error logs.")
elif current_status == "expired":
    print("\n⚠️ Batch expired without completing.")

BATCH STATUS: IN_PROGRESS
Batch ID: batch_69ac3b7c1d2481909a79bd49efedd1cf
Batch Name: wage_linkedin_exp_14

Progress:
  Total requests: 15000
  Completed: 1221
  Failed: 0
  Progress: 8.1%

Timestamps:
  Created: 2026-03-07 15:51:40
  Started: 2026-03-07 15:52:43

Current status: in_progress

⏳ Batch is still processing. Check back later.


## Download Results

Once the batch is completed, use this cell to download the results.

In [ ]:
# Replace with your batch ID if needed
# batch_id = "batch_xxxxxxxxxxxxx"

# Download results
output_dir = LINKEDIN_MATCHED_TO_SOEP_BLD / "openai_responses"

try:
    result = download_batch_results(batch_id, str(output_dir))
    print("\n✓ Results downloaded successfully!")
    print(f"Saved to: {result['output_path']}")
    print(f"Batch name: {result['batch_name']}")
    print(f"Number of responses: {result['n_responses']}")
except ValueError as e:
    print(f"\n❌ Error: {e}")
    print("Make sure the batch is completed before downloading.")